In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
application_train = pd.read_csv("../data/raw/application_train.csv")

In [5]:
bureau = pd.read_csv("../data/raw/bureau.csv")

In [6]:
bureau_balance = pd.read_csv(
    "../data/raw/bureau_balance.csv",
    usecols=[
        "SK_ID_BUREAU",
        "MONTHS_BALANCE",
        "STATUS"
    ],
    dtype={
        "SK_ID_BUREAU":"int32",
        "MONTHS_BALANCE":"int16",
        "STATUS":"category"
    }
)

bureau_balance.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype   
---  ------          -----   
 0   SK_ID_BUREAU    int32   
 1   MONTHS_BALANCE  int16   
 2   STATUS          category
dtypes: category(1), int16(1), int32(1)
memory usage: 182.2 MB


In [7]:
previous_application = pd.read_csv("../data/raw/previous_application.csv")

In [8]:
pos_cash = pd.read_csv("../data/raw/POS_CASH_balance.csv")



In [9]:
installments = pd.read_csv("../data/raw/installments_payments.csv")



In [10]:
credit_card = pd.read_csv("../data/raw/credit_card_balance.csv")

In [11]:
print("Shape :", bureau_balance.shape)

bureau_balance.head()

Shape : (27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [12]:
print("Shape :", bureau_balance.shape)

bureau_balance.head()

Shape : (27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [13]:
bureau_balance["STATUS"].value_counts().sort_index()

STATUS
0     7499507
1      242347
2       23419
3        8924
4        5847
5       62406
C    13646993
X     5810482
Name: count, dtype: int64

In [14]:
monthly_history = (

    bureau_balance

    .groupby("SK_ID_BUREAU")

    .size()

    .reset_index(name="MONTH_RECORDS")

)

monthly_history.head()

,SK_ID_BUREAU,MONTH_RECORDS
0,5001709,97
1,5001710,83
2,5001711,4
3,5001712,19
4,5001713,22


In [15]:
bureau_balance_num = (

    bureau_balance

    .groupby("SK_ID_BUREAU")

    .agg({

        "MONTHS_BALANCE":[

            "min",

            "max",

            "count"

        ]

    })

)

bureau_balance_num.columns = [

    "_".join(col)

    for col in bureau_balance_num.columns

]

bureau_balance_num.reset_index(inplace=True)

bureau_balance_num.head()

,SK_ID_BUREAU,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_count
0,5001709,-96,0,97
1,5001710,-82,0,83
2,5001711,-3,0,4
3,5001712,-18,0,19
4,5001713,-21,0,22


In [16]:
bureau_balance_status = (

    bureau_balance

    .groupby("SK_ID_BUREAU")

    .agg({

        "STATUS":[

            "nunique"

        ]

    })

)

bureau_balance_status.columns = [

    "STATUS_UNIQUE"

]

bureau_balance_status.reset_index(inplace=True)

bureau_balance_status.head()

,SK_ID_BUREAU,STATUS_UNIQUE
0,5001709,2
1,5001710,3
2,5001711,2
3,5001712,2
4,5001713,1


In [17]:
bureau_balance_agg = bureau_balance_num.merge(

    bureau_balance_status,

    on="SK_ID_BUREAU",

    how="left"

)

bureau_balance_agg.head()

,SK_ID_BUREAU,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_count,STATUS_UNIQUE
0,5001709,-96,0,97,2
1,5001710,-82,0,83,3
2,5001711,-3,0,4,2
3,5001712,-18,0,19,2
4,5001713,-21,0,22,1


In [18]:
print("Original Rows :", len(bureau_balance))

print("Aggregated Rows :", len(bureau_balance_agg))

print("Unique Bureau IDs :", bureau_balance["SK_ID_BUREAU"].nunique())

Original Rows : 27299925
Aggregated Rows : 817395
Unique Bureau IDs : 817395


In [19]:
from pathlib import Path

output_dir = Path("../data/processed")

output_dir.mkdir(parents=True, exist_ok=True)

In [20]:
bureau_balance_agg.to_csv(
    output_dir / "bureau_balance_agg.csv",
    index=False
)

print("bureau_balance_agg.csv saved successfully!")

bureau_balance_agg.csv saved successfully!


In [21]:
del bureau_balance

import gc

gc.collect()

0

In [22]:
bureau = bureau.merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left",
    validate="one_to_one"
)

print("Merged Bureau Shape:", bureau.shape)
bureau.head()

Merged Bureau Shape: (1716428, 21)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_count,STATUS_UNIQUE
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN,NaN,NaN,NaN,NaN


In [23]:
print(bureau.columns.tolist())

['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY', 'MONTHS_BALANCE_min', 'MONTHS_BALANCE_max', 'MONTHS_BALANCE_count', 'STATUS_UNIQUE']


In [24]:
def aggregate_bureau(df):

    aggregation = {

        "DAYS_CREDIT": ["min", "max", "mean", "std"],

        "CREDIT_DAY_OVERDUE": ["max", "mean", "sum"],

        "AMT_CREDIT_SUM": ["sum", "mean", "max"],

        "AMT_CREDIT_SUM_DEBT": ["sum", "mean", "max"],

        "AMT_CREDIT_SUM_OVERDUE": ["sum", "mean"],

        "AMT_ANNUITY": ["mean", "max"],

        "CNT_CREDIT_PROLONG": ["sum"],

        "MONTHS_BALANCE_count": ["mean", "max"],

        "STATUS_UNIQUE": ["mean", "max"]

    }

    bureau_customer = df.groupby("SK_ID_CURR").agg(aggregation)

    bureau_customer.columns = [

        "BUREAU_" + "_".join(col).upper()

        for col in bureau_customer.columns

    ]

    bureau_customer.reset_index(inplace=True)

    return bureau_customer

In [25]:
bureau_customer = aggregate_bureau(bureau)

print(bureau_customer.shape)

bureau_customer.head()

(305811, 23)


,SK_ID_CURR,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_STD,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_MAX,BUREAU_CNT_CREDIT_PROLONG_SUM,BUREAU_MONTHS_BALANCE_COUNT_MEAN,BUREAU_MONTHS_BALANCE_COUNT_MAX,BUREAU_STATUS_UNIQUE_MEAN,BUREAU_STATUS_UNIQUE_MAX
0,100001,-1572,-49,-735.000000,489.942514,0,0.0,0,1453365.000,207623.571429,378000.0,596686.5,85240.928571,373239.0,0.0,0.0,3545.357143,10822.5,0,24.571429,52.0,2.428571,3.0
1,100002,-1437,-103,-874.000000,431.451040,0,0.0,0,865055.565,108131.945625,450000.0,245781.0,49156.200000,245781.0,0.0,0.0,0.000000,0.0,0,13.750000,22.0,3.250000,4.0
2,100003,-2586,-606,-1400.750000,909.826128,0,0.0,0,1017400.500,254350.125000,810000.0,0.0,0.000000,0.0,0.0,0.0,NaN,NaN,0,NaN,NaN,NaN,NaN
3,100004,-1326,-408,-867.000000,649.124025,0,0.0,0,189037.800,94518.900000,94537.8,0.0,0.000000,0.0,0.0,0.0,NaN,NaN,0,NaN,NaN,NaN,NaN
4,100005,-373,-62,-190.666667,162.297053,0,0.0,0,657126.000,219042.000000,568800.0,568408.5,189469.500000,543087.0,0.0,0.0,1420.500000,4261.5,0,7.000000,13.0,2.000000,3.0


In [26]:
application_train = application_train.merge(

    bureau_customer,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"

)

print(application_train.shape)

(307511, 144)


In [27]:
del bureau
del bureau_customer

gc.collect()

0

In [28]:
print(previous_application.shape)

previous_application.head()

(1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
date_columns = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]

for col in date_columns:
    previous_application[col] = previous_application[col].replace(365243, np.nan)

In [30]:
def aggregate_previous_application(df):

    aggregation = {

        "AMT_APPLICATION": [
            "sum",
            "mean",
            "max"
        ],

        "AMT_CREDIT": [
            "sum",
            "mean",
            "max"
        ],

        "AMT_ANNUITY": [
            "mean",
            "max"
        ],

        "AMT_DOWN_PAYMENT": [
            "mean",
            "sum"
        ],

        "RATE_DOWN_PAYMENT": [
            "mean"
        ],

        "CNT_PAYMENT": [
            "mean",
            "max"
        ],

        "DAYS_DECISION": [
            "min",
            "max",
            "mean"
        ]

    }

    previous_customer = (

        df.groupby("SK_ID_CURR")

        .agg(aggregation)

    )

    previous_customer.columns = [

        "PREV_" + "_".join(col).upper()

        for col in previous_customer.columns

    ]

    previous_customer.reset_index(inplace=True)

    return previous_customer

In [31]:
previous_customer = aggregate_previous_application(previous_application)

print(previous_customer.shape)

previous_customer.head()

(338857, 17)


,SK_ID_CURR,PREV_AMT_APPLICATION_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_SUM,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_DOWN_PAYMENT_SUM,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_DAYS_DECISION_MEAN
0,100001,24835.5,24835.50,24835.5,23787.0,23787.00,23787.0,3951.000,3951.000,2520.0,2520.0,0.104326,8.0,8.0,-1740,-1740,-1740.0
1,100002,179055.0,179055.00,179055.0,179055.0,179055.00,179055.0,9251.775,9251.775,0.0,0.0,0.000000,24.0,24.0,-606,-606,-606.0
2,100003,1306309.5,435436.50,900000.0,1452573.0,484191.00,1035882.0,56553.990,98356.995,3442.5,6885.0,0.050030,10.0,12.0,-2341,-746,-1305.0
3,100004,24282.0,24282.00,24282.0,20106.0,20106.00,20106.0,5357.250,5357.250,4860.0,4860.0,0.212008,4.0,4.0,-815,-815,-815.0
4,100005,44617.5,22308.75,44617.5,40153.5,20076.75,40153.5,4813.200,4813.200,4464.0,4464.0,0.108964,12.0,12.0,-757,-315,-536.0


In [32]:
duplicates = previous_customer["SK_ID_CURR"].duplicated().sum()

print(f"Duplicate Customers: {duplicates}")

Duplicate Customers: 0


In [33]:
application_train = application_train.merge(

    previous_customer,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"

)

print(application_train.shape)

(307511, 160)


In [34]:
del previous_application
del previous_customer

gc.collect()

0

In [35]:
print(pos_cash.shape)

pos_cash.head()

(10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [36]:
print(pos_cash.isnull().sum())

SK_ID_PREV                   0
SK_ID_CURR                   0
MONTHS_BALANCE               0
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
NAME_CONTRACT_STATUS         0
SK_DPD                       0
SK_DPD_DEF                   0
dtype: int64


In [37]:
def aggregate_pos_cash(df):

    aggregation = {

        "MONTHS_BALANCE": [
            "min",
            "max",
            "mean"
        ],

        "CNT_INSTALMENT": [
            "mean",
            "max"
        ],

        "CNT_INSTALMENT_FUTURE": [
            "mean",
            "max"
        ],

        "SK_DPD": [
            "sum",
            "mean",
            "max"
        ],

        "SK_DPD_DEF": [
            "sum",
            "mean",
            "max"
        ]

    }

    pos_customer = (

        df.groupby("SK_ID_CURR")

        .agg(aggregation)

    )

    pos_customer.columns = [

        "POS_" + "_".join(col).upper()

        for col in pos_customer.columns

    ]

    pos_customer.reset_index(inplace=True)

    return pos_customer

In [38]:
pos_customer = aggregate_pos_cash(pos_cash)

print(pos_customer.shape)

pos_customer.head()

(337252, 14)


,SK_ID_CURR,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_MEAN,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_SUM,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_SUM,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX
0,100001,-96,-53,-72.555556,4.000000,4.0,1.444444,4.0,7,0.777778,7,7,0.777778,7
1,100002,-19,-1,-10.000000,24.000000,24.0,15.000000,24.0,0,0.000000,0,0,0.000000,0
2,100003,-77,-18,-43.785714,10.107143,12.0,5.785714,12.0,0,0.000000,0,0,0.000000,0
3,100004,-27,-24,-25.500000,3.750000,4.0,2.250000,4.0,0,0.000000,0,0,0.000000,0
4,100005,-25,-15,-20.000000,11.700000,12.0,7.200000,12.0,0,0.000000,0,0,0.000000,0


In [39]:
duplicates = pos_customer["SK_ID_CURR"].duplicated().sum()

print(f"Duplicate Customers : {duplicates}")

Duplicate Customers : 0


In [40]:
application_train = application_train.merge(

    pos_customer,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"

)

print(application_train.shape)

(307511, 173)


In [41]:
del pos_cash
del pos_customer

gc.collect()

0

In [42]:
print(application_train.shape)

(307511, 173)


In [43]:
[pos for pos in application_train.columns if pos.startswith("POS_")]

['POS_MONTHS_BALANCE_MIN',
 'POS_MONTHS_BALANCE_MAX',
 'POS_MONTHS_BALANCE_MEAN',
 'POS_CNT_INSTALMENT_MEAN',
 'POS_CNT_INSTALMENT_MAX',
 'POS_CNT_INSTALMENT_FUTURE_MEAN',
 'POS_CNT_INSTALMENT_FUTURE_MAX',
 'POS_SK_DPD_SUM',
 'POS_SK_DPD_MEAN',
 'POS_SK_DPD_MAX',
 'POS_SK_DPD_DEF_SUM',
 'POS_SK_DPD_DEF_MEAN',
 'POS_SK_DPD_DEF_MAX']

In [44]:
print(installments.shape)

installments.head()

(13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [45]:
installments["PAYMENT_DIFF"] = (
    installments["AMT_INSTALMENT"] -
    installments["AMT_PAYMENT"]
)

installments["PAYMENT_RATIO"] = (
    installments["AMT_PAYMENT"] /
    (installments["AMT_INSTALMENT"] + 1e-6)
)

installments["DAYS_LATE"] = (
    installments["DAYS_ENTRY_PAYMENT"] -
    installments["DAYS_INSTALMENT"]
)

installments["PAID_LATE"] = (
    installments["DAYS_LATE"] > 0
).astype("int8")

In [46]:
def aggregate_installments(df):

    aggregation = {

        "AMT_INSTALMENT": [
            "sum",
            "mean",
            "max"
        ],

        "AMT_PAYMENT": [
            "sum",
            "mean",
            "max"
        ],

        "PAYMENT_DIFF": [
            "mean",
            "max"
        ],

        "PAYMENT_RATIO": [
            "mean",
            "max"
        ],

        "DAYS_LATE": [
            "mean",
            "max"
        ],

        "PAID_LATE": [
            "sum",
            "mean"
        ]

    }

    installment_customer = (

        df.groupby("SK_ID_CURR")

        .agg(aggregation)

    )

    installment_customer.columns = [

        "INST_" + "_".join(col).upper()

        for col in installment_customer.columns

    ]

    installment_customer.reset_index(inplace=True)

    return installment_customer

In [47]:
installment_customer = aggregate_installments(installments)

print(installment_customer.shape)

installment_customer.head()

(339587, 15)


,SK_ID_CURR,INST_AMT_INSTALMENT_SUM,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_MAX,INST_AMT_PAYMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_MAX,INST_PAYMENT_DIFF_MEAN,INST_PAYMENT_DIFF_MAX,INST_PAYMENT_RATIO_MEAN,INST_PAYMENT_RATIO_MAX,INST_DAYS_LATE_MEAN,INST_DAYS_LATE_MAX,INST_PAID_LATE_SUM,INST_PAID_LATE_MEAN
0,100001,41195.925,5885.132143,17397.900,41195.925,5885.132143,17397.900,0.0,0.0,1.0,1.0,-7.285714,11.0,1,0.142857
1,100002,219625.695,11559.247105,53093.745,219625.695,11559.247105,53093.745,0.0,0.0,1.0,1.0,-20.421053,-12.0,0,0.000000
2,100003,1618864.650,64754.586000,560835.360,1618864.650,64754.586000,560835.360,0.0,0.0,1.0,1.0,-7.160000,-1.0,0,0.000000
3,100004,21288.465,7096.155000,10573.965,21288.465,7096.155000,10573.965,0.0,0.0,1.0,1.0,-7.666667,-3.0,0,0.000000
4,100005,56161.845,6240.205000,17656.245,56161.845,6240.205000,17656.245,0.0,0.0,1.0,1.0,-23.555556,1.0,1,0.111111


In [48]:
duplicates = installment_customer["SK_ID_CURR"].duplicated().sum()

print(f"Duplicate Customers : {duplicates}")

Duplicate Customers : 0


In [49]:
application_train = application_train.merge(

    installment_customer,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"

)

print(application_train.shape)

(307511, 187)


In [50]:
del installments
del installment_customer

gc.collect()

0

In [51]:
print(credit_card.shape)

credit_card.head()

(3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,11925.0,11925.0,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [52]:
credit_card.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE

In [53]:
def aggregate_credit_card(df):

    aggregation = {

        "AMT_BALANCE": [
            "sum",
            "mean",
            "max"
        ],

        "AMT_CREDIT_LIMIT_ACTUAL": [
            "mean",
            "max"
        ],

        "AMT_DRAWINGS_CURRENT": [
            "sum",
            "mean"
        ],

        "AMT_PAYMENT_TOTAL_CURRENT": [
            "sum",
            "mean"
        ],

        "CNT_DRAWINGS_CURRENT": [
            "sum",
            "mean"
        ],

        "SK_DPD": [
            "sum",
            "mean",
            "max"
        ],

        "SK_DPD_DEF": [
            "sum",
            "mean",
            "max"
        ]

    }

    credit_customer = (

        df.groupby("SK_ID_CURR")

        .agg(aggregation)

    )

    credit_customer.columns = [

        "CC_" + "_".join(col).upper()

        for col in credit_customer.columns

    ]

    credit_customer.reset_index(inplace=True)

    return credit_customer

In [54]:
credit_customer = aggregate_credit_card(credit_card)

print(credit_customer.shape)

credit_customer.head()

(103558, 18)


,SK_ID_CURR,CC_AMT_BALANCE_SUM,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_SUM,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_CNT_DRAWINGS_CURRENT_SUM,CC_CNT_DRAWINGS_CURRENT_MEAN,CC_SK_DPD_SUM,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_SUM,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX
0,100006,0.000,0.000000,0.00,270000.000000,270000,0.0,0.000000,0.000,0.000000,0,0.000000,0,0.000000,0,0,0.000000,0
1,100011,4031676.225,54482.111149,189000.00,164189.189189,180000,180000.0,2432.432432,334485.000,4520.067568,4,0.054054,0,0.000000,0,0,0.000000,0
2,100013,1743352.245,18159.919219,161420.22,131718.750000,157500,571500.0,5953.125000,654448.545,6817.172344,23,0.239583,1,0.010417,1,1,0.010417,1
3,100021,0.000,0.000000,0.00,675000.000000,675000,0.0,0.000000,0.000,0.000000,0,0.000000,0,0.000000,0,0,0.000000,0
4,100023,0.000,0.000000,0.00,135000.000000,225000,0.0,0.000000,0.000,0.000000,0,0.000000,0,0.000000,0,0,0.000000,0


In [55]:
duplicates = credit_customer["SK_ID_CURR"].duplicated().sum()

print(f"Duplicate Customers : {duplicates}")

Duplicate Customers : 0


In [56]:
application_train = application_train.merge(

    credit_customer,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"

)

print(application_train.shape)

(307511, 204)


In [57]:
del credit_card
del credit_customer

gc.collect()

0

In [58]:
print("=" * 60)

print("FINAL DATASET SHAPE")

print("=" * 60)

print(application_train.shape)

print("=" * 60)

print("Duplicate Customers")

print(application_train["SK_ID_CURR"].duplicated().sum())

print("=" * 60)

FINAL DATASET SHAPE
(307511, 204)
Duplicate Customers
0


In [59]:
missing_report = (

    application_train

    .isnull()

    .sum()

    .reset_index()

)

missing_report.columns = [

    "Feature",

    "Missing_Count"

]

missing_report["Missing_%"] = (

    missing_report["Missing_Count"]

    / len(application_train)

    * 100

)

missing_report.sort_values(

    "Missing_%",

    ascending=False,

    inplace=True
)

missing_report.head(20)

,Feature,Missing_Count,Missing_%
138,BUREAU_AMT_ANNUITY_MAX,227502,73.981744
137,BUREAU_AMT_ANNUITY_MEAN,227502,73.981744
192,CC_AMT_DRAWINGS_CURRENT_SUM,220606,71.739222
193,CC_AMT_DRAWINGS_CURRENT_MEAN,220606,71.739222
196,CC_CNT_DRAWINGS_CURRENT_SUM,220606,71.739222
197,CC_CNT_DRAWINGS_CURRENT_MEAN,220606,71.739222
198,CC_SK_DPD_SUM,220606,71.739222
199,CC_SK_DPD_MEAN,220606,71.739222
200,CC_SK_DPD_MAX,220606,71.739222
201,CC_SK_DPD_DEF_SUM,220606,71.739222


In [60]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [61]:
application_train.to_csv(

    output_dir / "application_train_merged.csv",

    index=False

)

print("application_train_merged.csv saved successfully!")

application_train_merged.csv saved successfully!
